# 1D network particle tracking on the Delaware River Basin

Passive tracer demo driven by pywatershed's network hydraulics export. The input file is produced by
`examples/02a_network_hydraulics_export.ipynb` in the pywatershed repository; set its path below.

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection

from fluvial_particle import (FileHydraulicsProvider, Network, NetworkConfig, estimate_particles,
                              run_network_simulation)

DRB = pathlib.Path("/home/rmcd/projects/pywatershed/examples/02a_network_hydraulics_export/drb_network_hydraulics.nc")
OUT = pathlib.Path("./network-drb-demo-output")
TRENTON = 4205

## 1. The network

In [ ]:
prov = FileHydraulicsProvider(DRB)
net = Network(prov.static, crs_wkt=prov.crs_wkt)
print(f"{net.n_reach} reaches, {net.outlets().size} outlets, {net.headwaters().size} headwaters, "
      f"{net.length.sum()/1000:.0f} km; {prov.times[0]} .. {prov.times[-1]}")
last = prov.hydraulics(prov.times[-1])
fig, ax = plt.subplots(figsize=(7, 9))
lc = LineCollection([np.column_stack(p) for p in net.polylines()], array=last["velocity"], cmap="viridis", linewidths=1.2)
ax.add_collection(lc); ax.autoscale(); ax.set_aspect("equal"); fig.colorbar(lc, label="velocity (m/s), last day")
ax.set_title("DRB network"); plt.show()

## 2. Sources: a slug at every headwater plus a week-long loading on the mainstem

In [ ]:
start = np.datetime64("1979-03-01")
heads = net.headwaters()
mainstem = [r for r in net.upstream_of(TRENTON)][20]  # a mainstem reach well upstream of Trenton
sources = [{"reach_id": int(r), "form": "slug", "time": 0.0, "mass": 100.0} for r in heads]
sources.append({"reach_id": int(mainstem), "form": "loading", "rate": 0.05, "start": "1979-03-03", "end": "1979-03-10"})
cfg = NetworkConfig.from_dict({
    "hydraulics_file": str(DRB), "start_time": str(start), "end_time": str(start + np.timedelta64(30, "D")),
    "dt": 900.0, "output_interval": 3600.0, "particle_mass": 1.0, "mass_units": "kg",
    "dispersion": {"model": "fischer"}, "sources": sources, "seed": 42,
})
budget = estimate_particles(cfg, prov, target_per_bin=50, bin_length=500.0)
print(budget.tail(3)); print("total particles at 1 kg/particle:", int(budget.particles.sum()))

## 3. Run one month

In [ ]:
res = run_network_simulation(cfg, OUT)
print(res.summary())

## 4. Arrival-time distributions at the outlets

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for outlet in net.outlets():
    h = res.arrival_histogram(int(outlet), bin_seconds=6 * 3600.0)
    if h.mass.sum() > 0:
        ax.plot(h.time, h.mass, label=f"reach {outlet}" + (" (Trenton)" if outlet == TRENTON else ""))
ax.set_ylabel("mass arriving per 6 h (kg)"); ax.legend(); ax.set_title("Breakthrough at the outlets")
released = res.positions(-1).mass.sum(); recovered = res.arrival_times().mass.sum()
print(f"released {released:.0f} kg, recovered at outlets {recovered:.0f} kg ({100*recovered/released:.0f}%)")

## 5. Concentration along the mainstem

In [ ]:
main_ids = net.upstream_of(TRENTON)
# walk the mainstem from Trenton upstream following the largest-flow parent
path_idx = [net.index_of(TRENTON)]
while True:
    parents = net.parents(path_idx[-1])
    if parents.size == 0:
        break
    path_idx.append(int(parents[np.argmax(last["flow_out"][parents])]))
path_idx = path_idx[::-1]
bins = res.bins(500.0)
sel = np.isin(bins.bin_reach, path_idx)
order = np.argsort([path_idx.index(r) for r in bins.bin_reach[sel]], kind="stable")
cube = res.concentration(None, bin_length=500.0, smoothing="auto").values[:, sel][:, order]
dist = np.cumsum(bins.bin_width[sel][order]) / 1000.0
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(dist, res.times, cube, shading="nearest", cmap="magma")
fig.colorbar(im, label="concentration (kg/m3)"); ax.set_xlabel("distance along mainstem (km)"); plt.show()

## 6. Map animation

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
ax.add_collection(LineCollection([np.column_stack(p) for p in net.polylines()], colors="lightgray", linewidths=0.8))
ax.autoscale(); ax.set_aspect("equal")
scat = ax.scatter([], [], s=4, c="crimson")
title = ax.set_title("")
frames = range(0, res.times.size, 3)

def update(i):
    df = res.map_positions(int(i))
    ok = df.status == 1
    scat.set_offsets(np.column_stack([df.x[ok], df.y[ok]]))
    title.set_text(str(res.times[i])[:16])
    return scat, title

anim = FuncAnimation(fig, update, frames=frames, interval=80, blit=False)
anim.save(OUT / "drb_particles.gif", writer="pillow", fps=12)
plt.close(fig)
from IPython.display import Image
Image(filename=str(OUT / "drb_particles.gif"))

In [ ]:
res.close(); prov.close()